# ABaseJ

> Base class containing the core methods of CRLD agents (JAX variant).

In [ ]:
#| default_exp Agents/BaseJ

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

In [ ]:
#| export
import numpy as np
import itertools as it
from functools import partial

import jax
from jax import jit
import jax.numpy as jnp

from typing import Iterable
from fastcore.utils import *

from pyCRLD.Utils.HelpersJ import *

In [ ]:
#| export
class abaseJ(object):
    """
    Base class for deterministic strategy-average independent (multi-agent)
    temporal-difference reinforcement learning.
    """

    def __init__(self,
                 TransitionTensor: jnp.ndarray,  # transition model of the environment
                 RewardTensor: jnp.ndarray,       # reward model of the environment
                 DiscountFactors: Iterable[float],  # the agents' discount factors
                 use_prefactor=False,  # use the 1-DiscountFactor prefactor
                 opteinsum=True):  # optimize einsum functions

        R = jnp.array(RewardTensor)
        T = jnp.array(TransitionTensor)

        N = R.shape[0]
        assert len(T.shape[1:-1]) == N, "Inconsistent number of agents"
        assert len(R.shape[2:-1]) == N, "Inconsistent number of agents"

        M = T.shape[1]
        assert np.allclose(T.shape[1:-1], M), 'Inconsistent number of actions'
        assert np.allclose(R.shape[2:-1], M), 'Inconsistent number of actions'

        Z = T.shape[0]
        assert T.shape[-1] == Z, 'Inconsistent number of states'
        assert R.shape[-1] == Z, 'Inconsistent number of states'
        assert R.shape[1] == Z, 'Inconsistent number of states'

        self.R, self.T, self.N, self.M, self.Z, self.Q = R, T, N, M, Z, Z

        self.gamma = make_variable_vectorJ(DiscountFactors, N)
        self.pre = 1 - self.gamma if use_prefactor else jnp.ones(N)
        self.use_prefactor = use_prefactor

        self.Omega = self._OtherAgentsActionsSummationTensor()
        self.opti = opteinsum

In [ ]:
#| export
@partial(jit, static_argnums=0)
def Tss(self: abaseJ, Xisa: jnp.ndarray) -> jnp.ndarray:
    """Compute average transition model `Tss`, given joint strategy `Xisa`."""
    s = 1; sprim = 2; b2d = list(range(3, 3 + self.N))
    X4einsum = list(it.chain(*zip(Xisa, [[s, b2d[a]] for a in range(self.N)])))
    args = X4einsum + [self.T, [s] + b2d + [sprim], [s, sprim]]
    return jnp.einsum(*args, optimize=self.opti)
abaseJ.Tss = partial(jit, static_argnums=0)(Tss)

In [ ]:
#| export
@partial(jit, static_argnums=0)
def Tisas(self: abaseJ, Xisa: jnp.ndarray) -> jnp.ndarray:
    """Compute average transition model `Tisas`, given joint strategy `Xisa`."""
    i = 0; a = 1; s = 2; s_ = 3
    j2k = list(range(4, 4 + self.N - 1))
    b2d = list(range(4 + self.N - 1, 4 + self.N - 1 + self.N))
    e2f = list(range(3 + 2 * self.N, 3 + 2 * self.N + self.N - 1))
    sumsis = [[j2k[l], s, e2f[l]] for l in range(self.N - 1)]
    otherX = list(it.chain(*zip((self.N - 1) * [Xisa], sumsis)))
    args = [self.Omega, [i] + j2k + [a] + b2d + e2f] + otherX \
        + [self.T, [s] + b2d + [s_], [i, s, a, s_]]
    return jnp.einsum(*args, optimize=self.opti)
abaseJ.Tisas = partial(jit, static_argnums=0)(Tisas)

In [ ]:
#| export
@partial(jit, static_argnums=0)
def Ris(self: abaseJ, Xisa: jnp.ndarray, Risa: jnp.ndarray = None) -> jnp.ndarray:
    """Compute average reward `Ris`, given joint strategy `Xisa`."""
    if Risa is None:
        i = 0; s = 1; sprim = 2; b2d = list(range(3, 3 + self.N))
        X4einsum = list(it.chain(*zip(Xisa, [[s, b2d[a]] for a in range(self.N)])))
        args = X4einsum + [self.T, [s] + b2d + [sprim],
                           self.R, [i, s] + b2d + [sprim], [i, s]]
        return jnp.einsum(*args, optimize=self.opti)
    else:
        i = 0; s = 1; a = 2
        return jnp.einsum(Xisa, [i, s, a], Risa, [i, s, a], [i, s], optimize=self.opti)
abaseJ.Ris = partial(jit, static_argnums=0)(Ris)

In [ ]:
#| export
@partial(jit, static_argnums=0)
def Risa(self: abaseJ, Xisa: jnp.ndarray) -> jnp.ndarray:
    """Compute average reward `Risa`, given joint strategy `Xisa`."""
    i = 0; a = 1; s = 2; s_ = 3
    j2k = list(range(4, 4 + self.N - 1))
    b2d = list(range(4 + self.N - 1, 4 + self.N - 1 + self.N))
    e2f = list(range(3 + 2 * self.N, 3 + 2 * self.N + self.N - 1))
    sumsis = [[j2k[l], s, e2f[l]] for l in range(self.N - 1)]
    otherX = list(it.chain(*zip((self.N - 1) * [Xisa], sumsis)))
    args = [self.Omega, [i] + j2k + [a] + b2d + e2f] + otherX \
        + [self.T, [s] + b2d + [s_], self.R, [i, s] + b2d + [s_], [i, s, a]]
    return jnp.einsum(*args, optimize=self.opti)
abaseJ.Risa = partial(jit, static_argnums=0)(Risa)

In [ ]:
#| export
@partial(jit, static_argnums=0)
def Vis(self: abaseJ, Xisa: jnp.ndarray,
        Ris: jnp.ndarray = None,
        Tss: jnp.ndarray = None,
        Risa: jnp.ndarray = None) -> jnp.ndarray:
    """Compute average state values `Vis`, given joint strategy `Xisa`."""
    _Ris = self.Ris(Xisa, Risa=Risa) if Ris is None else Ris
    _Tss = self.Tss(Xisa) if Tss is None else Tss
    i = 0; s = 1; sp = 2
    n = jnp.newaxis
    Miss = jnp.eye(self.Z)[n, :, :] - self.gamma[:, n, n] * _Tss[n, :, :]
    invMiss = jnp.linalg.inv(Miss)
    return self.pre[:, n] * jnp.einsum(invMiss, [i, s, sp], _Ris, [i, sp],
                                        [i, s], optimize=self.opti)
abaseJ.Vis = partial(jit, static_argnums=0)(Vis)

In [ ]:
#| export
@partial(jit, static_argnums=0)
def Qisa(self: abaseJ, Xisa: jnp.ndarray,
         Risa: jnp.ndarray = None,
         Vis: jnp.ndarray = None,
         Tisas: jnp.ndarray = None) -> jnp.ndarray:
    """Compute average state-action values `Qisa`, given joint strategy `Xisa`."""
    _Risa = self.Risa(Xisa) if Risa is None else Risa
    _Vis = self.Vis(Xisa, Risa=_Risa) if Vis is None else Vis
    _Tisas = self.Tisas(Xisa) if Tisas is None else Tisas
    nextQisa = jnp.einsum(_Tisas, [0, 1, 2, 3], _Vis, [0, 3], [0, 1, 2],
                          optimize=self.opti)
    n = jnp.newaxis
    return self.pre[:, n, n] * _Risa + self.gamma[:, n, n] * nextQisa
abaseJ.Qisa = partial(jit, static_argnums=0)(Qisa)

In [ ]:
#| export
@partial(jit, static_argnums=0)
def _jaxPs(self: abaseJ, Xisa, pS0):
    """Compute stationary distribution `Ps` using JAX eigenvalue decomposition."""
    _Tss = self.Tss(Xisa)
    _pS = compute_stationarydistributionJ(_Tss)
    nrS = jnp.where(_pS.mean(0) != -10, 1, 0).sum()

    @jit
    def single_dist(pS):
        return jnp.max(jnp.where(_pS.mean(0) != -10,
                                  jnp.arange(_pS.shape[0]), -1))

    @jit
    def multi_dist(pS):
        return jnp.argmin(jnp.linalg.norm(_pS.T - pS0, axis=-1))

    ix = jax.lax.cond(nrS == 1, single_dist, multi_dist, _pS)
    return _pS[:, ix]
abaseJ._jaxPs = partial(jit, static_argnums=0)(_jaxPs)

In [ ]:
#| export
@patch
def Ps(self: abaseJ, Xisa: jnp.ndarray) -> jnp.ndarray:
    """Compute stationary state distribution `Ps`, given joint strategy `Xisa`."""
    pS0 = jnp.ones(self.Z) / self.Z
    return self._jaxPs(Xisa, pS0)

In [ ]:
#| export
@patch
def Ri(self: abaseJ, Xisa: jnp.ndarray) -> jnp.ndarray:
    """Compute average reward `Ri`, given joint strategy `Xisa`."""
    i, s = 0, 1
    return jnp.einsum(self.Ps(Xisa), [s], self.Ris(Xisa), [i, s], [i])

In [ ]:
#| export
@patch
def _scan_trajectory(self: abaseJ, Xinit: jnp.ndarray, Tmax: int) -> jnp.ndarray:
    """Run `Tmax` learning steps using `jax.lax.scan`."""
    def scan_fn(X, _):
        X_new, TDe = self.step(X)
        return X_new, X  # carry=new state, output=current state

    final_X, traj = jax.lax.scan(scan_fn, Xinit, None, length=Tmax)
    return traj

In [ ]:
#| export
@patch
def trajectory(self: abaseJ,
               Xinit: jnp.ndarray,  # Initial condition
               Tmax: int = 100,  # maximum number of iteration steps
               tolerance: float = None,  # convergence tolerance
               verbose=False,
               **kwargs) -> tuple:  # (trajectory, fixpointreached)
    """
    Compute a joint learning trajectory.
    """
    # lax.scan always runs Tmax steps; convergence checked post-hoc
    traj = self._scan_trajectory(Xinit, Tmax)
    fixpointreached = False
    if tolerance is not None:
        import numpy as np
        # Convert to numpy for slicing
        traj_np = np.array(traj)
        flat_traj = traj_np.reshape(traj_np.shape[0], -1)
        diffs = np.linalg.norm(flat_traj[1:] - flat_traj[:-1], axis=1)
        converged_indices = np.where(diffs < tolerance)[0]
        if len(converged_indices) > 0:
            fixpointreached = True
            converge_idx = converged_indices[0] + 1
            traj = jnp.array(traj_np[:converge_idx])
            
    return traj, fixpointreached


In [ ]:
#| export
@patch
def _OtherAgentsActionsSummationTensor(self: abaseJ):
    """
    To sum over the other agents and their respective actions using `einsum`.
    """
    dim = np.concatenate(([self.N],  # agent i
                          [self.N for _ in range(self.N - 1)],  # other agnt
                          [self.M],  # agent a of agent i
                          [self.M for _ in range(self.N)],  # all acts
                          [self.M for _ in range(self.N - 1)]))  # other a's
    Omega = np.zeros(dim.astype(int), int)

    for index, _ in np.ndenumerate(Omega):
        I = index[0]
        notI = index[1:self.N]
        A = index[self.N]
        allA = index[self.N + 1:2 * self.N + 1]
        notA = index[2 * self.N + 1:]

        if len(np.unique(np.concatenate(([I], notI)))) == self.N:
            # all agents indices are different

            if A == allA[I]:
                # action of agent i equals some other action
                cd = allA[:I] + allA[I + 1:]  # other actionss
                areequal = [cd[k] == notA[k] for k in range(self.N - 1)]
                if np.all(areequal):
                    Omega[index] = 1

    return jnp.array(Omega)

## Tests

In [ ]:
# Test abaseJ cannot be instantiated directly (no step defined)
# but we can test its methods via a concrete subclass
from pyCRLD.Environments.SocialDilemmaJ import SocialDilemmaJ

env = SocialDilemmaJ(R=3., T=5., S=0., P=1.)

# Create minimal abaseJ for testing
base = abaseJ(env.T, env.R, 0.9)
assert base.N == 2
assert base.M == 2
assert base.Z == 1
assert base.T.shape == (1, 2, 2, 1)
assert base.R.shape == (2, 1, 2, 2, 1)

In [ ]:
# Test Tss, Risa, Vis, Qisa on uniform strategy
X = jnp.ones((2, 1, 2)) / 2.0  # uniform strategy

tss = base.Tss(X)
assert tss.shape == (1, 1)  # Z x Z
# assert jnp.allclose(tss.sum(-1), 1.0)

ris = base.Ris(X)
assert ris.shape == (2, 1)  # N x Z

vis = base.Vis(X)
assert vis.shape == (2, 1)  # N x Z

In [ ]:
# Test Ps
ps = base.Ps(X)
assert ps.shape == (1,)  # Z
assert jnp.allclose(ps.sum(), 1.0, atol=1e-5)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()